# Raster value extraction to validation plots (points)

Extract values from tiled raster datasets (local or S3) to point locations 
stored in geopackage files. This notebook handles:

- Multiple input point files
- Time series raster extraction
- Tiled raster datasets with spatial indexing
- Automatic reprojection and spatial joins
- Timestamped output files

--------------------------------------------------------------------------------
Author Information
--------------------------------------------------------------------------------
Name                    | Affiliation
------------------------|-------------------------------------------------------
Paul Montesano, Phd     | NASA GSFC, ESSIC

--------------------------------------------------------------------------------
Last Updated: 2026-05-12
--------------------------------------------------------------------------------

In [1]:
# 1. IMPORTS AND SETUP
# ============================================================================
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio.env import Env
import shapely
from shapely.geometry import Point
import warnings
warnings.filterwarnings('ignore')

### Functions

In [2]:
# 2. CORE EXTRACTION FUNCTIONS
# ============================================================================

def do_point_query(point_geom, raster_path, band=1):
    """Extract raster value at point location."""
    with rio.open(raster_path) as src:
        values = list(src.sample([point_geom.coords[0]], indexes=band))
        return values[0][0] if values else np.nan

def get_transformation(raster_path, gdf):
    """Get coordinate transformation from GDF CRS to raster CRS."""
    with rio.open(raster_path) as src:
        from functools import partial
        import pyproj
        from shapely.ops import transform as shapely_transform
        
        project = pyproj.Transformer.from_crs(
            gdf.crs, src.crs, always_xy=True
        ).transform
        
        return project

def reproject_gdf_to_raster(gdf, raster_path):
    """Reproject entire GDF to match raster CRS."""
    with rio.open(raster_path) as src:
        return gdf.to_crs(src.crs)

def spatial_join_footprints(points_gdf, footprint_path, raster_dict):
    """
    Spatially join points with raster tile footprints to get paths.
    
    Parameters:
    -----------
    points_gdf : GeoDataFrame
        Points to extract values for
    footprint_path : str
        Path to footprint geopackage
    raster_dict : dict
        Configuration dictionary
        
    Returns:
    --------
    GeoDataFrame with added path columns (s3_path or local_path)
    """
    
    # Read footprint
    footprint_gdf = gpd.read_file(footprint_path)
    
    print(f"  Footprint columns: {footprint_gdf.columns.tolist()}")
    
    # Ensure same CRS
    if points_gdf.crs != footprint_gdf.crs:
        points_gdf = points_gdf.to_crs(footprint_gdf.crs)
    
    # Drop any existing index_right columns from previous joins
    cols_to_drop = [col for col in points_gdf.columns if col.startswith('index_right') or col.startswith('index_left')]
    if cols_to_drop:
        points_gdf = points_gdf.drop(columns=cols_to_drop)
    
    # Identify the path column in the footprint
    path_col_in_footprint = None
    for col in ['s3_path', 'local_path', 'location', 'path', 'file']:
        if col in footprint_gdf.columns:
            path_col_in_footprint = col
            break
    
    if path_col_in_footprint is None:
        print(f"  Warning: No path column found in footprint")
        return points_gdf
    
    print(f"  Path column in footprint: '{path_col_in_footprint}'")
    
    # Keep only necessary columns from footprint
    keep_cols = ['geometry', path_col_in_footprint]
    footprint_gdf = footprint_gdf[keep_cols]
    
    # Spatial join
    joined_gdf = gpd.sjoin(points_gdf, footprint_gdf, how='left', predicate='intersects')
    
    # Drop index_right immediately
    if 'index_right' in joined_gdf.columns:
        joined_gdf = joined_gdf.drop(columns=['index_right'])
    
    # Remove duplicate columns (keep first occurrence)
    joined_gdf = joined_gdf.loc[:, ~joined_gdf.columns.duplicated()]
    
    return joined_gdf


def extract_from_tiled_rasters(gdf, raster_dict, band=None):
    """
    Extract values from tiled rasters to points.
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Points with path column pointing to raster tiles
    raster_dict : dict
        Configuration dictionary with optional 'multi_band' flag
    band : int or None
        Specific band to extract. If None and multi_band=True, extracts all bands
    
    Returns:
    --------
    GeoDataFrame with new column(s) containing extracted values
    """
    
    location = raster_dict['location']
    data_name = raster_dict['data_name']
    data_source = raster_dict.get('data_source', 'unknown')
    multi_band = raster_dict.get('multi_band', False)
    
    print(f"Extracting from {data_name}...")
    
    # Identify the path column from the footprint
    possible_path_cols = ['s3_path', 'local_path', 'location', 'path', 'file']
    path_col = None
    
    for col in possible_path_cols:
        if col in gdf.columns:
            path_col = col
            break
    
    if path_col is None:
        print(f"  Warning: No path column found. Skipping extraction.")
        gdf[data_name] = np.nan
        return gdf
    
    print(f"  Using path column: '{path_col}'")
    
    # Subset to points that have a valid path
    subset_gdf = gdf[gdf[path_col].notna()].copy()
    n_with_path = len(subset_gdf)
    n_total = len(gdf)
    
    print(f"  Points with valid paths: {n_with_path}/{n_total}")
    
    if n_with_path == 0:
        print(f"  Warning: No points have valid paths. Adding NaN column.")
        gdf[data_name] = np.nan
        return gdf
    
    # Get band information from first raster
    first_raster_path = subset_gdf.iloc[0][path_col]
    if location == 'local' and raster_dict.get('data_dir'):
        first_raster_path = os.path.join(raster_dict['data_dir'], first_raster_path)
    
    # Set up S3 session if needed
    if location == 's3':
        session = Env(AWS_NO_SIGN_REQUEST='YES')
    else:
        session = None
    
    # Determine bands to extract
    try:
        if session:
            with session:
                with rio.open(first_raster_path) as src:
                    n_bands = src.count
                    band_descriptions = [src.descriptions[i-1] if src.descriptions[i-1] else f'b{i}' 
                                       for i in range(1, n_bands + 1)]
        else:
            with rio.open(first_raster_path) as src:
                n_bands = src.count
                band_descriptions = [src.descriptions[i-1] if src.descriptions[i-1] else f'b{i}' 
                                   for i in range(1, n_bands + 1)]
        
        print(f"  Raster has {n_bands} band(s): {band_descriptions}")
        
    except Exception as e:
        print(f"  Warning: Could not read band info from first raster: {e}")
        n_bands = 1
        band_descriptions = ['b1']
    
    # Determine which bands to extract
    if band is not None:
        # Extract specific band
        bands_to_extract = [band]
        band_names = [band_descriptions[band-1] if band <= len(band_descriptions) else f'b{band}']
    elif multi_band and n_bands > 1:
        # Extract all bands
        bands_to_extract = list(range(1, n_bands + 1))
        band_names = band_descriptions
    else:
        # Default: extract band 1
        bands_to_extract = [1]
        band_names = [band_descriptions[0] if band_descriptions else 'b1']
    
    print(f"  Extracting bands: {bands_to_extract} ({band_names})")
    
    # Extract values for each band
    for band_idx, band_num in enumerate(bands_to_extract):
        band_name = band_names[band_idx]
        
        # Clean band name for column name (remove spaces, special chars)
        clean_band_name = band_name.replace(' ', '_').replace('/', '_').replace('\\', '_')
        col_name = f"{data_name}_{clean_band_name}" if len(bands_to_extract) > 1 else data_name
        
        print(f"    Extracting band {band_num} ({band_name})...")
        
        values = []
        n_success = 0
        n_error = 0
        
        for idx, row in subset_gdf.iterrows():
            raster_path = row[path_col]
            
            # Add base directory for local paths
            if location == 'local' and raster_dict.get('data_dir'):
                raster_path = os.path.join(raster_dict['data_dir'], raster_path)
            
            try:
                if session:
                    with session:
                        # Reproject point geometry to raster CRS
                        transform_func = get_transformation(raster_path, gdf)
                        reprojected_point = shapely.ops.transform(transform_func, row.geometry)
                        
                        # Extract value
                        value = do_point_query(reprojected_point, raster_path, band=band_num)
                        values.append(value)
                        n_success += 1
                else:
                    # Reproject point geometry to raster CRS
                    transform_func = get_transformation(raster_path, gdf)
                    reprojected_point = shapely.ops.transform(transform_func, row.geometry)
                    
                    # Extract value
                    value = do_point_query(reprojected_point, raster_path, band=band_num)
                    values.append(value)
                    n_success += 1
                    
            except Exception as e:
                if n_error < 5:  # Only print first 5 errors
                    print(f"      Error at index {idx}: {e}")
                values.append(np.nan)
                n_error += 1
        
        # Add values to subset
        subset_gdf[col_name] = values
        
        print(f"    Successfully extracted: {n_success}/{n_with_path}")
        if n_error > 0:
            print(f"    Errors: {n_error}")
    
    # Merge back to full GDF
    new_cols = [col for col in subset_gdf.columns if col.startswith(data_name)]
    gdf = gdf.merge(subset_gdf[new_cols], left_index=True, right_index=True, how='left')
    
    # Drop the path column after extraction
    if path_col in gdf.columns:
        gdf = gdf.drop(columns=[path_col])
    
    return gdf

# %%
# 6. SAVE RESULTS
# ============================================================================

def save_extracted_data(gdf, input_path, driver='GPKG', include_timestamp=True):
    """
    Save extracted data to file.
    
    Parameters:
    -----------
    gdf : GeoDataFrame
        Data to save
    input_path : str
        Input file path (used as base for output filename)
    driver : str
        Output format driver (default='GPKG')
    include_timestamp : bool
        Add date stamp to filename (default=True)
    
    Returns:
    --------
    str : Path to saved file
    """
    import datetime
    
    # Parse output path
    base_dir = os.path.dirname(input_path)
    base_name = os.path.basename(input_path)
    name, ext = os.path.splitext(base_name)
    
    # Add date if requested
    if include_timestamp:
        date_str = datetime.datetime.now().strftime('%Y%m%d')
        output_file = os.path.join(base_dir, f"{name}_{date_str}{ext}")
    else:
        output_file = input_path
    
    # Create directory if needed
    os.makedirs(base_dir, exist_ok=True)
    
    # Save (overwrite if exists)
    print(f"\nSaving results to: {output_file}")
    print(f"  Shape: {gdf.shape}")
    print(f"  Driver: {driver}")
    
    if os.path.exists(output_file):
        print(f"  ⚠ File exists - overwriting")
    
    gdf.to_file(output_file, driver=driver)
    print("  ✓ Saved successfully!")
    
    return output_file

# %%
# 3. CONFIGURATION - DEFINE YOUR RASTER PRODUCTS
# ============================================================================

def create_raster_config(product_name, product_desc, tindex_path=None, footprint_path=None,
                         location='s3', data_dir=None):
    """Helper function to create raster configuration dictionary."""
    return {
        'PRODUCT_NAME': product_name,
        'PRODUCT_DESC': product_desc,
        'TINDEX_MASTER_FN': tindex_path,
        'location': location,
        'footprint_fn': footprint_path,
        'data_name': f'value_{product_name}',
        'data_source': 'maap-s3' if location == 's3' else 'local',
        'data_dir': data_dir,
        'drop_cols_list': ['file', 'tile_num', 'tile_version', 'tile_group', 
                          'map_version', 'Unnamed: 0', 's3_path', 'local_path', 
                          'creation time', 'index_right']
    }



In [4]:
# Build list of raster dictionaries
# Start with base template for common parameters
d = {
    'location': 's3',
    'data_source': 'maap-s3',
    'multi_band': True,
    'data_dir': None,
    'drop_cols_list': ['file', 'tile_num', 'tile_version', 'tile_group', 
                      'map_version', 'Unnamed: 0', 's3_path', 'local_path', 
                      'creation time', 'index_right']
}

# Define specific product dictionaries
dict_tcctrend = { 
    # TerraPulse unclipped annual boreal TCC
    'location': 's3',
    'data_dir': None,
    'file_name': None,
    'multi_band': True,
    'is_tiled': True,
    'footprint_fn': '/projects/my-public-bucket/DPS_tile_lists/TCCTREND/build_stack_v2023_2/TCCTREND_TP_2020/TCCTREND_tindex_master.gpkg',
    'drop_cols_list': ['footprint_name', 'path', 'file', 'area_km2', 'area_ha', 'tile_num', 's3_path'],
    's3_url_prefix': None,
    'data_source': 'terrapulse-s3',
    'data_name': 'value_tcctrend2020',
    'PRODUCT_NAME': 'TCCTREND2020',
    'PRODUCT_DESC': 'TerraPulse TCC Trend 2020',
    'aws_credential_fn': None
}

dict_tcctrendpval = { 
    # TerraPulse unclipped annual boreal TCC
    'location': 's3',
    'data_dir': None,
    'file_name': None,
    'multi_band': True,
    'is_tiled': True,
    'footprint_fn': '/projects/my-public-bucket/DPS_tile_lists/TCCTRENDPVAL/build_stack_v2023_2/TCCTRENDPVAL_TP_2020/TCCTREND_tindex_master.gpkg',
    'drop_cols_list': ['footprint_name', 'path', 'file', 'area_km2', 'area_ha', 'tile_num', 's3_path'],
    's3_url_prefix': None,
    'data_source': 'terrapulse-s3',
    'data_name': 'value_tcctrend2020pval',
    'PRODUCT_NAME': 'TCCTREND2020PVAL',
    'PRODUCT_DESC': 'TerraPulse TCC Trend P-value 2020',
    'aws_credential_fn': None
}

dict_agbtrend = { 
    # 2020-2025 AGB trends (ols)
    'location': 's3',
    'data_dir': None,
    'file_name': None,
    'multi_band': True,
    'is_tiled': True,
    'footprint_fn': '/projects/my-public-bucket/DPS_tile_lists/TRENDOLS/compute_trends_ols_v3/AGB_v3.1_multiyr_2020-2025/TRENDOLS_tindex_master.gpkg',
    'drop_cols_list': ['footprint_name', 'path', 'file', 'area_km2', 'area_ha', 'tile_num', 's3_path'],
    's3_url_prefix': None,
    'data_source': 'maap-s3',
    'data_name': 'value_AGB_v3.1_multiyr_2020-2025',
    'PRODUCT_NAME': 'AGB_v3.1_multiyr_2020-2025',
    'PRODUCT_DESC': 'AGB v3.1 Trends (OLS) 2020-2025',
    'aws_credential_fn': None
}

dict_cacc2020 = { 
    # 2020 carbon accumulation
    'location': 's3',
    'data_dir': None,
    'file_name': None,
    'multi_band': True,
    'is_tiled': True,
    'footprint_fn': '/projects/my-public-bucket/DPS_tile_lists/CACC/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/CACC_tindex_master.gpkg',
    'drop_cols_list': ['footprint_name', 'path', 'file', 'area_km2', 'area_ha', 'tile_num', 's3_path'],
    's3_url_prefix': None,
    'data_source': 'maap-s3',
    'data_name': 'value_CACC_2020_v3.1_multiyr',
    'PRODUCT_NAME': 'CACC_2020_v3.1_multiyr',
    'PRODUCT_DESC': 'Carbon Accumulation 2020 from v3.1_multiyr',
    'aws_credential_fn': None
}

# Build your list of raster configurations
LIST_DICTS = []

# Add TCC Trend
LIST_DICTS.append(dict_tcctrend)
LIST_DICTS.append(dict_tcctrendpval)
LIST_DICTS.append(dict_agbtrend)
LIST_DICTS.append(dict_cacc2020)

LIST_DICTS += [{**d, 
                'PRODUCT_DESC': f'Boreal AGB v3.1 from H30 {YEAR} with multiyr model & ATL08 v6',
                'PRODUCT_NAME': f'AGB_H30_{YEAR}_v3.1_multiyr',
                'TINDEX_MASTER_FN': f'/projects/shared-buckets/montesano/DPS_tile_lists/BOREAL_MAP/v3.1.0/AGB_H30_{YEAR}/full_run_niter30_multiyear_atl08v6/AGB_tindex_master.csv',
                'footprint_fn': f'/projects/my-public-bucket/databank/footprints/footprints_maap_AGB_H30_{YEAR}_v3.1_multiyr-s3.gpkg',
                'data_name': f'value_AGB_H30_{YEAR}_v3.1_multiyr',
                'multi_band': True
               }
               for YEAR in range(2016, 2026)]

LIST_DICTS += [{**d, 
                'PRODUCT_DESC': f'Boreal Height v3.1 from H30 {YEAR} with multiyr model & ATL08 v6',
                'PRODUCT_NAME': f'Ht_H30_{YEAR}_v3.1_multiyr',
                'TINDEX_MASTER_FN': f'/projects/shared-buckets/montesano/DPS_tile_lists/BOREAL_MAP/v3.1.0/Ht_H30_{YEAR}/full_run_niter30_multiyear_atl08v6/HT_tindex_master.csv',
                'footprint_fn': f'/projects/my-public-bucket/databank/footprints/footprints_maap_Ht_H30_{YEAR}_v3.1_multiyr-s3.gpkg',
                'data_name': f'value_Ht_H30_{YEAR}_v3.1_multiyr',
                'multi_band': True
               }
               for YEAR in range(2016, 2026)]

In [5]:
print(f"Configured {len(LIST_DICTS)} raster products:")
for i, cfg in enumerate(LIST_DICTS):  
    print(f"  {i+1}. {cfg['PRODUCT_NAME']}: {cfg['PRODUCT_DESC']}")


Configured 24 raster products:
  1. TCCTREND2020: TerraPulse TCC Trend 2020
  2. TCCTREND2020PVAL: TerraPulse TCC Trend P-value 2020
  3. AGB_v3.1_multiyr_2020-2025: AGB v3.1 Trends (OLS) 2020-2025
  4. CACC_2020_v3.1_multiyr: Carbon Accumulation 2020 from v3.1_multiyr
  5. AGB_H30_2016_v3.1_multiyr: Boreal AGB v3.1 from H30 2016 with multiyr model & ATL08 v6
  6. AGB_H30_2017_v3.1_multiyr: Boreal AGB v3.1 from H30 2017 with multiyr model & ATL08 v6
  7. AGB_H30_2018_v3.1_multiyr: Boreal AGB v3.1 from H30 2018 with multiyr model & ATL08 v6
  8. AGB_H30_2019_v3.1_multiyr: Boreal AGB v3.1 from H30 2019 with multiyr model & ATL08 v6
  9. AGB_H30_2020_v3.1_multiyr: Boreal AGB v3.1 from H30 2020 with multiyr model & ATL08 v6
  10. AGB_H30_2021_v3.1_multiyr: Boreal AGB v3.1 from H30 2021 with multiyr model & ATL08 v6
  11. AGB_H30_2022_v3.1_multiyr: Boreal AGB v3.1 from H30 2022 with multiyr model & ATL08 v6
  12. AGB_H30_2023_v3.1_multiyr: Boreal AGB v3.1 from H30 2023 with multiyr model & 

In [6]:
# %%
# 4. LOAD INPUT POINTS - MULTIPLE FILES
# ============================================================================

# Define list of input points files
INPUT_POINTS_FILES = [
    '/projects/my-private-bucket/reference/eurasia_forest_structure_plots_agbd_20260513.gpkg',
     #'/projects/my-private-bucket/reference/eurasia_forest_structure_plots_smrytrees.gpkg',
     #'/projects/my-private-bucket/reference/KolymaRegion_all.gpkg',
     #'/projects/my-private-bucket/reference/Miesner_plots.gpkg',
     #'/projects/my-private-bucket/reference/nfi_plus_20220603.geojson'
]

# Or generate list programmatically
# INPUT_POINTS_FILES = glob.glob('/projects/my-private-bucket/validation_points_*.gpkg')

print(f"Found {len(INPUT_POINTS_FILES)} input files to process:")
for i, f in enumerate(INPUT_POINTS_FILES, 1):
    print(f"  {i}. {os.path.basename(f)}")

Found 1 input files to process:
  1. eurasia_forest_structure_plots_agbd_20260513.gpkg


In [7]:
# %%
# 5. EXTRACT VALUES FROM ALL RASTERS - LOOP THROUGH INPUT FILES
# ============================================================================

for file_idx, INPUT_POINTS_FILE in enumerate(INPUT_POINTS_FILES, 1):
    
    print(f"\n{'#'*70}")
    print(f"# PROCESSING INPUT FILE {file_idx}/{len(INPUT_POINTS_FILES)}")
    print(f"# {os.path.basename(INPUT_POINTS_FILE)}")
    print(f"{'#'*70}\n")
    
    # Load points
    print(f"Loading points from: {INPUT_POINTS_FILE}")
    points_gdf = gpd.read_file(INPUT_POINTS_FILE)
    
    # Ensure WGS84
    if points_gdf.crs != 'EPSG:4326':
        points_gdf = points_gdf.to_crs('EPSG:4326')
    
    print(f"Loaded {len(points_gdf)} points with {len(points_gdf.columns)} columns")
    print(f"CRS: {points_gdf.crs}")
    
    # Initialize results
    results_gdf = points_gdf.copy()
    
    # Extract from each raster
    for i, raster_config in enumerate(LIST_DICTS):
        print(f"\n{'='*60}")
        print(f"[{i+1}/{len(LIST_DICTS)}] Processing: {raster_config['PRODUCT_DESC']}")
        print(f"{'='*60}")
        
        # Spatial join with footprints (adds path column)
        results_gdf = spatial_join_footprints(
            results_gdf, 
            raster_config['footprint_fn'],
            raster_config
        )
        
        # Extract values (and remove path column after)
        results_gdf = extract_from_tiled_rasters(results_gdf, raster_config)
        
        # Extra cleanup: remove any columns that shouldn't be there
        unwanted_cols = [col for col in results_gdf.columns if any([
            col.startswith('index_'),
            col.startswith('tile_'),
            col.startswith('map_'),
            col.endswith('_left'),
            col.endswith('_right'),
            col in ['creation time', 's3_path', 'local_path', 'location', 'path']
        ])]
        
        if unwanted_cols:
            results_gdf = results_gdf.drop(columns=unwanted_cols)
    
    # ALL EXTRACTIONS COMPLETE FOR THIS FILE - NOW SAVE
    print(f"\n{'='*60}")
    print(f"EXTRACTION COMPLETE FOR {os.path.basename(INPUT_POINTS_FILE)}")
    print(f"{'='*60}")
    print(f"Final GDF shape: {results_gdf.shape}")
    print(f"Value columns added: {len([c for c in results_gdf.columns if c.startswith('value_')])}")
    
    # Save results using input filename as base (with timestamp)
    saved_file = save_extracted_data(
        results_gdf, 
        INPUT_POINTS_FILE,
        driver='GPKG',
        include_timestamp=True
    )
    
    print(f"\nOriginal file: {INPUT_POINTS_FILE}")
    print(f"Saved to:      {saved_file}")

print(f"\n{'#'*70}")
print(f"# ALL INPUT FILES PROCESSED!")
print(f"# Processed {len(INPUT_POINTS_FILES)} files")
print(f"{'#'*70}")

# %%
# 6. OPTIONAL: SUMMARY STATISTICS (for last file processed)
# ============================================================================

# Show extraction summary
value_cols = [col for col in results_gdf.columns if col.startswith('value_')]
print(f"\nExtracted {len(value_cols)} raster datasets:")
for col in value_cols[:10]:  # Show first 10
    n_valid = results_gdf[col].notna().sum()
    n_total = len(results_gdf)
    pct_valid = 100 * n_valid / n_total
    print(f"  {col}: {n_valid}/{n_total} ({pct_valid:.1f}%) valid values")

if len(value_cols) > 10:
    print(f"  ... and {len(value_cols) - 10} more")


######################################################################
# PROCESSING INPUT FILE 1/1
# eurasia_forest_structure_plots_agbd_20260513.gpkg
######################################################################

Loading points from: /projects/my-private-bucket/reference/eurasia_forest_structure_plots_agbd_20260513.gpkg
Loaded 646 points with 22 columns
CRS: EPSG:4326

[1/24] Processing: TerraPulse TCC Trend 2020
  Footprint columns: ['tile_num', 'tile_version', 'tile_group', 'map_version', 's3_path', 'local_path', 'geometry']
  Path column in footprint: 's3_path'
Extracting from value_tcctrend2020...
  Using path column: 's3_path'
  Points with valid paths: 530/646
  Raster has 1 band(s): ['slope_ols']
  Extracting bands: [1] (['slope_ols'])
    Extracting band 1 (slope_ols)...
    Successfully extracted: 530/530

[2/24] Processing: TerraPulse TCC Trend P-value 2020
  Footprint columns: ['tile_num', 'tile_version', 'tile_group', 'map_version', 's3_path', 'local_path', 'geom

In [8]:
results_gdf.head(300).tail()

,year,site,glas_campaign,glas_ndx,glas_shot,group_name,rad_m,num_plots,plot_area_m2,canopy_closure_perc,...,value_Ht_H30_2021_v3.1_multiyr_mean_ht,value_Ht_H30_2021_v3.1_multiyr_std_ht,value_Ht_H30_2022_v3.1_multiyr_mean_ht,value_Ht_H30_2022_v3.1_multiyr_std_ht,value_Ht_H30_2023_v3.1_multiyr_mean_ht,value_Ht_H30_2023_v3.1_multiyr_std_ht,value_Ht_H30_2024_v3.1_multiyr_mean_ht,value_Ht_H30_2024_v3.1_multiyr_std_ht,value_Ht_H30_2025_v3.1_multiyr_mean_ht,value_Ht_H30_2025_v3.1_multiyr_std_ht
295,2008,605488310_3,L3G,605488310,3.0,Kotuykan River,10.0,1.0,314.159265,NaN,...,7.297945,0.927605,8.439475,0.707223,8.892033,0.906209,8.245121,1.070870,8.551675,0.617760
296,2008,607398971_25,L3G,607398971,25.0,Kotuykan River,15.0,1.0,706.858347,NaN,...,4.605954,1.224532,5.559286,1.485969,3.970985,1.223228,4.188756,1.014392,3.630051,1.032018
297,2008,538613582_1,L3F,538613582,1.0,Kotuykan River,10.0,1.0,314.159265,NaN,...,8.260155,1.046790,7.830012,0.885648,8.114983,0.985579,8.837088,0.919236,8.754816,0.814441
298,2008,607398971_22,L3G,607398971,22.0,Kotuykan River,15.0,1.0,706.858347,NaN,...,4.890710,0.717679,6.613897,0.582835,5.860057,0.546806,6.791610,0.426700,6.626459,0.508387
299,2008,381762521_23,L3C,381762521,23.0,Kotuykan River,15.0,1.0,706.858347,NaN,...,5.212412,1.020661,5.698810,0.981531,3.431078,0.988723,5.060767,0.966752,6.003950,0.943441


In [10]:
results_gdf.columns

Index(['year', 'site', 'glas_campaign', 'glas_ndx', 'glas_shot', 'group_name',
       'rad_m', 'num_plots', 'plot_area_m2', 'canopy_closure_perc',
       'min_meas_dbh_cm', 'num_trees', 'agbd_v1_mg_ha', 'n_trees_with_biomass',
       'total_biomass_kg', 'mean_tree_biomass_kg', 'has_valid_area',
       'has_biomass', 'agbd_v2_mg_ha', 'total_biomass_mg', 'plot_radius_m',
       'geometry', 'value_tcctrend2020', 'value_tcctrend2020pval',
       'value_AGB_v3.1_multiyr_2020-2025_trendslope',
       'value_AGB_v3.1_multiyr_2020-2025_trendintercept',
       'value_AGB_v3.1_multiyr_2020-2025_r2',
       'value_AGB_v3.1_multiyr_2020-2025_trendslope_lo',
       'value_AGB_v3.1_multiyr_2020-2025_trendslope_hi',
       'value_AGB_v3.1_multiyr_2020-2025_kendall_tau',
       'value_AGB_v3.1_multiyr_2020-2025_kendall_pvalue',
       'value_CACC_2020_v3.1_multiyr_carbon_acc_mean',
       'value_CACC_2020_v3.1_multiyr_carbon_acc_std',
       'value_CACC_2020_v3.1_multiyr_carbon_acc_ci_lower',
       '